In [11]:
import copy
import time
import math
import random
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchvision.utils import make_grid
from scipy import linalg
from sklearn.model_selection import train_test_split

In [4]:
SEED    = 42
BATCH   = 128
ZDIM    = 64           # latent size of AE / Sparse AE / VAE / MLP feature
NZ      = 100          # noise size of GAN / DCGAN
LAMBDA  = 5e-5         # sparsity weight (Sparse AE)
T       = 1000         # number of diffusion steps
N_FID   = 5000         # number of generated images used for FID


EPOCHS = dict(
    mlp=40,
    dec=40,
    ae=40,
    sae=40,
    vae=40,
    gan=40,
    dcgan=40,
    ddpm=40,
)

PATIENCE = dict(
    mlp=5,
    dec=5,
    ae=5,
    sae=5,
    vae=5,
    ddpm=10,
)

MIN_DELTA = 1e-3 # minimum improvement for early stopping

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


seed_all()

# 1. DATA

In [18]:
train_raw = torchvision.datasets.FashionMNIST(root="data", train=True, download=True)
test_raw = torchvision.datasets.FashionMNIST(root="data", train=False, download=True)

In [19]:
X = torch.cat([train_raw.data, test_raw.data])
X = X.float() / 255
X = X.unsqueeze(1) # shape (70000, 1, 28, 28)
Y = torch.cat([train_raw.targets, test_raw.targets])

In [20]:
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=SEED,
    stratify=Y
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=Y_temp
)

In [21]:
train_loader = DataLoader(TensorDataset(X_train, Y_train), BATCH, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), BATCH, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, Y_test), BATCH, shuffle=False)

print("train", len(X_train), "| val", len(X_val), "| test", len(X_test))

train 56000 | val 7000 | test 7000


# 2. METRICS

In [ ]:
from torchmetrics import MeanSquaredError
from torchmetrics.image import (
    PeakSignalNoiseRatio,
    StructuralSimilarityIndexMeasure,
    FrechetInceptionDistance,
)

mse_metric = MeanSquaredError().to(dev)
psnr_metric = PeakSignalNoiseRatio(data_range=1.0, dim=(1, 2, 3)).to(dev)
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(dev)
fid_metric = FrechetInceptionDistance(
    feature=2048,
    reset_real_features=False,
    normalize=True,
).to(dev)

@torch.no_grad()
def recon_metrics(recon_fn):
    """Test MSE / PSNR / SSIM between recon_fn(x) and the original x."""
    mse_metric.reset()
    psnr_metric.reset()
    ssim_metric.reset()

    for x, _ in test_loader:
        x = x.to(dev)
        x_rec = recon_fn(x).clamp(0, 1)

        mse_metric.update(x_rec, x)
        psnr_metric.update(x_rec, x)
        ssim_metric.update(x_rec, x)

    return dict(
        mse=mse_metric.compute().item(),
        psnr=psnr_metric.compute().item(),
        ssim=ssim_metric.compute().item(),
    )


@torch.no_grad()
def add_images_to_fid(images, real):
    """Feed images (N, 1, 28, 28) in 0..1 to the FID metric. real=True for real images."""
    for start in range(0, len(images), 250):
        batch = images[start:start + 250].to(dev).clamp(0, 1)
        batch = batch.repeat(1, 3, 1, 1) 
        fid_metric.update(batch, real=real)


def fid_of(fake_images):
    fid_metric.reset()
    add_images_to_fid(fake_images, real=False)
    return fid_metric.compute().item()


def latency_ms(fn, runs=20):
    """Average time in milliseconds for one call of fn() (= one batch of 64 images)."""
    with torch.no_grad():
        fn()                                       # warm-up
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(runs):
            fn()
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
    return elapsed / runs * 1000